In [1]:
import os
import shutil
from tqdm import tqdm

### ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# static
COPY ./static/. /static/

# templates
COPY ./templates/. /templates/

# api
COPY api.py .

# app.py
COPY app.py .

# parser
COPY cls_parser.pkl .

# descriptions
COPY df_descriptions.csv .

# functions
COPY functions.py .

# functions_app
COPY functions_app.py .

# passwords
COPY passwords.py .

# preprocessing
COPY preprocessing.py .

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# expose port 5000
EXPOSE 5000

# run script when image is run
CMD ["python", "app.py"]

Writing Dockerfile


### Bring items to local directory

In [3]:
# files
list_str_file = [
    'api.py',
    'app.py',
    'cls_parser.pkl',
    'df_descriptions.csv',
    'functions.py',
    'functions_app.py',
    'passwords.py',
    'preprocessing.py',
    'requirements.txt',
]
for str_file in tqdm(list_str_file):
    str_source = f'../02_local_app/{str_file}'
    str_destination = f'./{str_file}'
    shutil.copy(str_source, str_destination)

100%|██████████| 9/9 [00:00<00:00, 311.71it/s]


In [4]:
# directories
list_str_dir = [
    'static',
    'templates',
]
for str_dir in tqdm(list_str_dir):
    str_source = f'../02_local_app/{str_dir}'
    str_destination = f'./{str_dir}'
    try:
        shutil.copytree(str_source, str_destination)
    except FileExistsError:
        pass

100%|██████████| 2/2 [00:00<00:00, 463.02it/s]


### Build and push

In [5]:
%%sh

# image name
image=gen-xii-comparison

# Get the account number associated with the current IAM credentials
account=$(aws sts get-caller-identity --query Account --output text)

# did we have an error?
if [ $? -ne 0 ]
then
    exit 255
fi

# Get the region defined in the current configuration (default to us-west-2 if none defined)
region=$(aws configure get region)
region=${region:-us-west-2}

# get destination of repo
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${image}" > /dev/null 2>&1

# if it doesnt exist...create it
if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${image}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

# build and add tag
docker build  -t ${image} .
docker tag ${image} ${fullname}
# push to ecr
docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
Sending build context to Docker daemon  5.043MB
Step 1/17 : FROM python:3.9
3.9: Pulling from library/python
bc0734b949dc: Pulling fs layer
b5de22c0f5cd: Pulling fs layer
917ee5330e73: Pulling fs layer
b43bd898d5fb: Pulling fs layer
7fad4bffde24: Pulling fs layer
cd0903c43c21: Pulling fs layer
b85288e0cb16: Pulling fs layer
7a97f6368ea6: Pulling fs layer
cd0903c43c21: Waiting
b43bd898d5fb: Waiting
b85288e0cb16: Waiting
7fad4bffde24: Waiting
7a97f6368ea6: Waiting
b5de22c0f5cd: Verifying Checksum
b5de22c0f5cd: Download complete
bc0734b949dc: Download complete
917ee5330e73: Verifying Checksum
917ee5330e73: Download complete
7fad4bffde24: Verifying Checksum
7fad4bffde24: Download complete
cd0903c43c21: Verifying Checksum
cd0903c43c21: Download complete
b85288e0cb16: Verifying Checksum
b85288e0cb16: Download complete
7a97f6368ea6: Download complete
bc0734b949dc: Pull complete
b5de22c0f5cd: Pull complete
917ee5330e73: Pull complete
b43bd898d5fb: Verifying Checksum
b43bd898d5f

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 5.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 6.5 MB/s eta 0:00:00
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warni

  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.7 MB/s eta 0:00:00
Removing intermediate container bac73d32091f
 ---> 5b00300d4762
Step 16/17 : EXPOSE 5000
 ---> Running in d57c59e931a8
Removing intermediate container d57c59e931a8
 ---> b95bc99b05bb
Step 17/17 : CMD ["python", "app.py"]
 ---> Running in 58e5202b97f4
Removing intermediate container 58e5202b97f4
 ---> 7289e35bdec8
Successfully built 7289e35bdec8
Successfully tagged gen-xii-comparison:latest
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xii-comparison]
186302d1862e: Preparing
6dfc25a9bc81: Preparing
a726ac0e4604: Preparing
8d57914d1631: Preparing
b3d6d98c94f7: Preparing
3b5846e7e3d9: Preparing
f27fe01b30f7: Preparing
10cc3205c79c: Preparing
728

### Clean-up

In [6]:
list_files = list_str_file + list_str_dir + ['Dockerfile']

for str_file in list_files:
    try:
        shutil.rmtree(str_file)
    except NotADirectoryError:
        os.remove(str_file)
    except FileNotFoundError:
        pass